# HIMNETRA - Part 1: Data → Deep-learning lake mapping → Risk score (Himachal Pradesh)

**How to use:** Runtime ▸ Change runtime type ▸ **T4 GPU**. Run cells **top to bottom, one at a time**. Each cell ends with ✅ (OK) or an `assert` that tells you what to fix.

| Cell | What it does | You must check |
|---|---|---|
| 1-2 | install, Drive, Earth Engine login | paste your **Project ID** in cell 1 |
| 3 | 21 hand-picked HP glacial lakes (HPSDMA/NDMA inventory) + South Lhonak back-test | - |
| 5 | snaps each seed to the real lake | 👀 look at the picture, list bad ones in cell 6 |
| 7-8 | downloads 8 yearly tiles per lake, quality-checks them | 👀 weak-label picture |
| 9 | trains U-Net and DeepLabv3+ (held-out lakes = honest IoU) | IoU values |
| 10-11 | area time series, polygons, risk score, frontend JSON files | table printed |
| 12 | charts + back-test + zip download | - |

Outputs (in Drive `himnetra/out/` and a zip): `lakes.json, risk.json, timeseries.json, polygons.geojson, segmentation.json, metrics.json, backtest.json, qc.csv`.


In [ ]:
#@title 1) Install + settings
!pip -q install segmentation-models-pytorch pyproj rasterio shapely

PROJECT = "YOUR_PROJECT_ID"  #@param {type:"string"}

YEARS   = list(range(2018, 2026))                 # one composite per year (Jul 15 - Sep 30)
BASE    = "/content/drive/MyDrive/himnetra"       # everything is saved here (survives Colab restarts)
HALF_PX = 224                                     # tile = 448 x 448 px @ 10 m (4.48 km square)
print("Settings loaded. PROJECT =", PROJECT)


In [ ]:
#@title 2) Mount Drive + start Earth Engine + test
import os, io, json, glob, math, time, random, warnings, datetime
import numpy as np
warnings.filterwarnings("ignore")
from google.colab import drive
drive.mount("/content/drive")
for d in ["tiles", "out", "review", "models"]:
    os.makedirs(f"{BASE}/{d}", exist_ok=True)

import ee
assert PROJECT and PROJECT != "YOUR_PROJECT_ID", "❌ Put your real Google Cloud Project ID in cell 1 and re-run it"
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()          # sign in with the SAME Google account that registered Earth Engine
    ee.Initialize(project=PROJECT)

elev = ee.Image("USGS/SRTMGL1_003").sample(ee.Geometry.Point([77.22, 32.52]), 30).first().get("elevation").getInfo()
assert elev and elev > 3000, f"❌ unexpected elevation {elev}"
print(f"✅ Earth Engine OK (DEM near Gepang Gath = {elev} m)")


In [ ]:
#@title 3) The lake list (hand-picked from the official HPSDMA / NDMA inventory)
# Source: HPSDMA "Potential Moraine Dammed Glacial Lakes in Himachal Himalayas" (LISS-III 2019) -> NDMA/SCCC codes,
#         plus Chandra Tal from the 2025 IIT-Mandi 'very high susceptibility' list.
# Columns: id, name, lat, lng, district, basin, snap_radius_m, expected_area_ha (2019 inventory), reference
# Coordinates are only SEEDS: cell 5 snaps each one to the real water body and shows you a picture to confirm.
LAKES = [
 ("GL-HP-001","Gepang Gath",           32.5248,77.2201,"Lahaul & Spiti","Chandra",2000, 92.0,"NDMA001"),
 ("GL-HP-002","Samudra Tapu",          32.4986,77.5464,"Lahaul & Spiti","Chandra",3000,132.5,"NDMA002"),
 ("GL-HP-003","Chandra Tal",           32.4806,77.6147,"Lahaul & Spiti","Chandra",2000,None,"IIT-Mandi 2025 list (approx. coords)"),
 ("GL-HP-004","Baspa-head lake",       31.3386,78.2531,"Kinnaur","Baspa",1500, 18.9,"NDMA081"),
 ("GL-HP-005","Kalpa NW lake",         31.6607,78.1669,"Kinnaur","Satluj",1500, 27.9,"NDMA082"),
 ("GL-HP-006","Taiti Garang",          31.5852,78.1854,"Kinnaur","Satluj",1200,  5.4,"NDMA433"),
 ("GL-HP-007","Yamuna-basin lake",     31.2603,78.2546,"Kinnaur / Shimla","Yamuna",1500, 17.3,"NDMA461"),
 ("GL-HP-008","Parvati lake (north)",  31.9144,77.5254,"Kullu","Parvati",1500, 12.5,"NDMA077"),
 ("GL-HP-009","Beas lake (Kullu)",     32.1569,77.2983,"Kullu","Beas",1200,  6.9,"NDMA432"),
 ("GL-HP-010","Beas lake (SCCC-34)",   32.2923,77.0788,"Kullu","Beas",1200,  5.4,"SCCC"),
 ("GL-HP-011","Beas lake (SCCC-38)",   32.1704,77.0409,"Kullu","Beas",1200,  7.0,"SCCC"),
 ("GL-HP-012","Parvati lake (SCCC-35)",32.1339,77.4559,"Kullu","Parvati",1200, 11.6,"SCCC"),
 ("GL-HP-013","Parvati lake (SCCC-43)",32.1798,77.4924,"Kullu","Parvati",1200,  5.7,"SCCC"),
 ("GL-HP-014","Beas lake (SCCC-28)",   31.8486,77.7897,"Kullu","Beas",1500, 14.3,"SCCC"),
 ("GL-HP-015","Ravi lake (Barot area)",32.2334,76.7533,"Mandi / Kangra","Ravi",1500, 10.3,"SCCC"),
 ("GL-HP-016","Ravi lake (SCCC-39)",   32.3356,76.3318,"Chamba / Kangra","Ravi",1500, 29.2,"SCCC"),
 ("GL-HP-017","Miyar lake (NDMA434)",  32.8423,76.5379,"Lahaul & Spiti","Miyar",1500, 13.4,"NDMA434"),
 ("GL-HP-018","Miyar lake (SCCC-45)",  32.9879,76.2362,"Lahaul & Spiti","Miyar",1200,  6.5,"SCCC"),
 ("GL-HP-019","Bhaga lake",            32.6308,77.3070,"Lahaul & Spiti","Bhaga",1200,  6.8,"NDMA435"),
 ("GL-HP-020","Chandra lake (NDMA436)",32.6044,77.6175,"Lahaul & Spiti","Chandra",1200, 5.6,"NDMA436"),
 ("GL-HP-021","Chandra lake (NDMA437)",32.7218,77.4128,"Lahaul & Spiti","Chandra",1500, 9.9,"NDMA437"),
]
# Outside Himachal. Used ONLY for the back-test slide (never for training / HP metrics).
BACKTEST = [("BT-SK-001","South Lhonak (Sikkim) - back-test only",27.9,88.18,"Sikkim","Teesta",2500,None,"2023 GLOF")]

# Held-out lakes (never seen in training) -> honest validation numbers
VAL_IDS = ["GL-HP-005", "GL-HP-012", "GL-HP-016"]
PRIORITY = {"GL-HP-001", "GL-HP-002", "GL-HP-003"}   # lakes named in the synopsis / IIT-Mandi 'very high' list
print(len(LAKES), "Himachal lakes +", len(BACKTEST), "back-test lake")
assert len({l[0] for l in LAKES}) == len(LAKES), "duplicate ids"


In [ ]:
#@title 4) Helper functions (just run)
from scipy import ndimage as ndi
from pyproj import Transformer
import matplotlib
import matplotlib.pyplot as plt

S2 = "COPERNICUS/S2_SR_HARMONIZED"
BANDS = ["B2", "B3", "B4", "B8", "B11", "NDWI", "VV", "DEM", "SLOPE"]   # stored channel order
C = {n: i for i, n in enumerate(BANDS)}
S3 = np.ones((3, 3), bool)

def epsg_for(lng):                       # UTM zone (43N..45N for HP/Sikkim)
    return 32600 + int((lng + 180) // 6) + 1

def haversine(lat1, lng1, lat2, lng2):
    p = math.pi / 180
    a = math.sin((lat2-lat1)*p/2)**2 + math.cos(lat1*p)*math.cos(lat2*p)*math.sin((lng2-lng1)*p/2)**2
    return 12742000 * math.asin(math.sqrt(a))

def _cloud_mask(im):
    scl = im.select("SCL")
    bad = scl.eq(8).Or(scl.eq(9)).Or(scl.eq(10))     # cloud + cirrus ONLY (class 3 'shadow' swallows dark lakes)
    return im.updateMask(bad.Not())

def build_image(roi, year):
    start, end = f"{year}-07-15", f"{year}-09-30"
    s2c = (ee.ImageCollection(S2).filterBounds(roi).filterDate(start, end)
           .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60)).map(_cloud_mask))
    names = ["B2", "B3", "B4", "B8", "B11"]
    s2 = ee.Image(ee.Algorithms.If(s2c.size().gt(0), s2c.median().select(names),
                                   ee.Image.constant([0, 0, 0, 0, 0]).rename(names)))
    refl = s2.unmask(0).divide(10000)
    ndwi = refl.normalizedDifference(["B3", "B8"]).rename("NDWI")
    s1c = (ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(roi).filterDate(start, end)
           .filter(ee.Filter.eq("instrumentMode", "IW")).select("VV"))
    vv = ee.Image(ee.Algorithms.If(s1c.size().gt(0), s1c.median(), ee.Image.constant(0))).rename("VV").unmask(0)
    dem = ee.Image("USGS/SRTMGL1_003").rename("DEM")
    slope = ee.Terrain.slope(dem).rename("SLOPE")
    return refl.addBands([ndwi, vv, dem, slope]).select(BANDS).toFloat()

def fetch(lat, lng, year, half_px=HALF_PX, scale=10):
    """Download one (2*half_px)^2 tile as float32 array [H, W, 9] on an exact UTM grid."""
    epsg = epsg_for(lng)
    X, Y = Transformer.from_crs(4326, epsg, always_xy=True).transform(lng, lat)
    n = 2 * half_px
    x0 = round(X / scale) * scale - half_px * scale
    y0 = round(Y / scale) * scale + half_px * scale
    roi = ee.Geometry.Rectangle([x0, y0 - n*scale, x0 + n*scale, y0], proj=f"EPSG:{epsg}", geodesic=False)
    req = dict(expression=build_image(roi, year), fileFormat="NPY",
               grid=dict(dimensions=dict(width=n, height=n),
                         affineTransform=dict(scaleX=scale, shearX=0, translateX=x0,
                                              shearY=0, scaleY=-scale, translateY=y0),
                         crsCode=f"EPSG:{epsg}"))
    arr = ee.data.computePixels(req)
    if isinstance(arr, (bytes, bytearray)): arr = np.load(io.BytesIO(arr))
    x = np.stack([arr[b] for b in BANDS], -1).astype("float32")
    return x, dict(epsg=int(epsg), x0=float(x0), y0=float(y0), scale=int(scale))

def with_retry(fn, tries=3, wait=6):
    for t in range(tries):
        try: return fn()
        except Exception as e:
            if t == tries - 1: raise
            time.sleep(wait * (t + 1))

def clean(mask, min_px):
    lab, k = ndi.label(mask, structure=S3)
    if k == 0: return mask
    sizes = np.bincount(lab.ravel()); keep = sizes >= min_px; keep[0] = False
    return keep[lab]

def fill_small_holes(m, max_px=100):
    holes = ndi.binary_fill_holes(m) & ~m
    lab, k = ndi.label(holes)
    if k == 0: return m
    sizes = np.bincount(lab.ravel()); ok = sizes <= max_px; ok[0] = False
    return m | ok[lab]

def strict_water(x):
    """Optical weak label: water is dark in NIR/SWIR; snow & steep shadow are rejected."""
    m = ((x[..., C["NDWI"]] > 0.05) & (x[..., C["B8"]] < 0.15) & (x[..., C["B11"]] < 0.10)
         & (x[..., C["B4"]] < 0.25) & (x[..., C["SLOPE"]] < 25) & (x[..., C["B3"]] > 0))   # SRTM slope is noisy on lakes
    m = ndi.binary_closing(m, iterations=1)
    m = fill_small_holes(m)
    return ndi.binary_opening(m)

def center_component(mask, cy, cx):
    """Connected component nearest to pixel (cy, cx); None if mask empty."""
    lab, k = ndi.label(mask, structure=S3)
    if k == 0: return None
    _, (iy, ix) = ndi.distance_transform_edt(lab == 0, return_indices=True)
    return lab == lab[iy[cy, cx], ix[cy, cx]]

def snap(lat, lng, r_m, year=2022):
    """Find the real lake near a seed coordinate. Returns (info|None, x, mask)."""
    x, meta = with_retry(lambda: fetch(lat, lng, year, half_px=250, scale=20))
    if (x[..., C["B3"]] > 0).mean() < 0.5: return None, x, None
    m = clean(strict_water(x), 4)
    lab, k = ndi.label(m, structure=S3)
    H, W = m.shape; cy = cx = 250
    yy, xx = np.mgrid[:H, :W]; d = np.hypot(yy - cy, xx - cx) * 20
    best = None
    for i in range(1, k + 1):
        sel = lab == i; ha = sel.sum() * 0.04; dmin = d[sel].min()
        if dmin <= r_m and ha >= 1.5:
            score = ha / (1 + dmin / 1000)
            if best is None or score > best[0]: best = (score, i, ha)
    if best is None: return None, x, m
    ys, xs = np.nonzero(lab == best[1])
    Y = meta["y0"] - (ys.mean() + .5) * 20; X = meta["x0"] + (xs.mean() + .5) * 20
    lng2, lat2 = Transformer.from_crs(meta["epsg"], 4326, always_xy=True).transform(X, Y)
    return dict(lat=lat2, lng=lng2, area_ha=best[2], moved_m=haversine(lat, lng, lat2, lng2)), x, m

def rgb(x, gain=3.0):
    return np.clip(x[..., [C["B4"], C["B3"], C["B2"]]] * gain, 0, 1)

print("✅ helpers ready")


In [ ]:
#@title 5) Snap every seed to the real lake + REVIEW PICTURE (look at it!)
from concurrent.futures import ThreadPoolExecutor
ALL_SEEDS = [(*l, False) for l in LAKES] + [(*l, True) for l in BACKTEST]

def _snap_job(l):
    lid, name, lat, lng, dist, basin, r, exp_ha, ref, bt = l
    try:
        info, x, m = snap(lat, lng, r)
        return lid, info, x, m, None
    except Exception as e:
        return lid, None, None, None, str(e)[:150]

with ThreadPoolExecutor(4) as ex:
    res = list(ex.map(_snap_job, ALL_SEEDS))

SNAP, PREVIEW, rows = {}, {}, []
for l, (lid, info, x, m, err) in zip(ALL_SEEDS, res):
    lid, name, lat, lng, dist, basin, r, exp_ha, ref, bt = l
    if err:            status = "ERROR"
    elif info is None: status = "NO_WATER"
    elif exp_ha and not (0.35 <= info["area_ha"] / exp_ha <= 3.0): status = "CHECK_SIZE"
    else:              status = "OK"
    if info: SNAP[lid] = info
    if x is not None: PREVIEW[lid] = (rgb(x), m)
    rows.append((lid, name[:28], status, round(info["area_ha"], 1) if info else None, exp_ha,
                 round(info["moved_m"]) if info else None, err or ""))

print(f"{'id':10} {'name':28} {'status':11} {'found_ha':>8} {'inventory_ha':>12} {'moved_m':>8}")
for r_ in rows: print(f"{r_[0]:10} {r_[1]:28} {r_[2]:11} {str(r_[3]):>8} {str(r_[4]):>12} {str(r_[5]):>8}  {r_[6]}")

ids = [r_[0] for r_ in rows if r_[0] in PREVIEW]
cols = 4; nr = math.ceil(len(ids) / cols)
fig, axs = plt.subplots(nr, cols, figsize=(4.2 * cols, 4.2 * nr)); axs = np.atleast_1d(axs).ravel()
for ax in axs: ax.axis("off")
for ax, lid in zip(axs, ids):
    im, m = PREVIEW[lid]; ax.imshow(im)
    if m is not None and m.any(): ax.contour(m, levels=[.5], colors="red", linewidths=.8)
    ax.plot(250, 250, "c+", ms=10)          # cyan + = your seed coordinate
    st = [r_[2] for r_ in rows if r_[0] == lid][0]
    ax.set_title(f"{lid} {st}", fontsize=9)
plt.tight_layout(); plt.savefig(f"{BASE}/review/01_snap_review.png", dpi=110); plt.show()
json.dump(SNAP, open(f"{BASE}/out/snapped_lakes.json", "w"), indent=1)
print("\n👀 Red outline = detected water. Cyan + = seed. Each outline should sit on a real lake (dark blue/green/milky water),")
print("   NOT on a glacier, shadow or river. Write bad ids in EXCLUDE in the next cell.")


In [ ]:
#@title 6) Decide which lakes to keep
EXCLUDE = ["GL-HP-014", "GL-HP-017"]    #@param {type:"raw"}   ids that looked wrong in the picture
KEEP = [l for l in LAKES if l[0] in SNAP and l[0] not in EXCLUDE]
BT   = [l for l in BACKTEST if l[0] in SNAP]
dropped = [l[0] for l in LAKES if l[0] not in {k[0] for k in KEEP}]
print("Keeping", len(KEEP), "Himachal lakes. Dropped:", dropped)
assert len(KEEP) >= 8, "❌ Fewer than 8 lakes survived - loosen thresholds in strict_water() or check the picture"
assert sum(l[0] in VAL_IDS for l in KEEP) >= 1, "❌ none of VAL_IDS survived: edit VAL_IDS in cell 3"
print("✅ lake list ready")


In [ ]:
#@title 7) Download all tiles (about 5-10 min; safe to re-run, skips finished files)
TILES = f"{BASE}/tiles"
jobs = [(l[0], y) for l in KEEP + BT for y in YEARS]

def tile_job(a):
    lid, y = a
    p = f"{TILES}/{lid}_{y}.npz"
    if os.path.exists(p): return lid, y, "cached", ""
    try:
        s = SNAP[lid]
        x, meta = with_retry(lambda: fetch(s["lat"], s["lng"], y))
        np.savez_compressed(p, x=x, **meta)
        return lid, y, "ok", ""
    except Exception as e:
        return lid, y, "FAIL", str(e)[:120]

t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    out = list(ex.map(tile_job, jobs))
fails = [o for o in out if o[2] == "FAIL"]
print(f"{len(out)-len(fails)}/{len(out)} tiles ready in {time.time()-t0:.0f}s ; files on disk:", len(glob.glob(f'{TILES}/*.npz')))
for f in fails[:10]: print("  FAIL", f)
t = np.load(glob.glob(f"{TILES}/*.npz")[0]); print("tile shape:", t["x"].shape, "(expect (448, 448, 9))")
assert t["x"].shape == (2*HALF_PX, 2*HALF_PX, 9), "❌ wrong tile shape"
assert len(fails) < 0.25 * len(out), "❌ too many failures - re-run this cell once (rate-limit), then paste the error to me"
print("✅ export OK")


In [ ]:
#@title 8) Quality-check every tile + REVIEW PICTURE
import csv
QC = {}
for p in sorted(glob.glob(f"{TILES}/*.npz")):
    d = np.load(p); x = d["x"]; lid, y = os.path.basename(p)[:-4].rsplit("_", 1)
    opt = float((x[..., C["B3"]] > 0).mean()); sar = float((x[..., C["VV"]] != 0).mean())
    comp = center_component(clean(strict_water(x), 8), HALF_PX, HALF_PX)
    ha = float(comp.sum() * 0.01) if comp is not None else 0.0
    if comp is not None and ndi.distance_transform_edt(~comp)[HALF_PX, HALF_PX] > 30: ha = 0.0   # nearest water is >300 m away
    flag = "NO_OPTICAL" if opt < 0.9 else ("NO_WATER" if ha < 1.0 else "OK")
    QC[(lid, int(y))] = dict(path=p, optical=opt, sar=sar, label_ha=ha, flag=flag)
with open(f"{BASE}/out/qc.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["lake", "year", "optical_cover", "sar_cover", "label_ha", "flag"])
    for (a, b), v in sorted(QC.items()): w.writerow([a, b, round(v["optical"], 2), round(v["sar"], 2), round(v["label_ha"], 1), v["flag"]])
from collections import Counter
print("tile flags:", dict(Counter(v["flag"] for v in QC.values())))
print("tiles per lake that are OK (need >=3 for a trend):")
okc = Counter(k[0] for k, v in QC.items() if v["flag"] == "OK")
print({k: okc.get(k, 0) for k in [l[0] for l in KEEP + BT]})

pick = random.Random(0).sample([k for k, v in QC.items() if v["flag"] == "OK"], 6)
fig, axs = plt.subplots(3, 6, figsize=(20, 10))
for j, k in enumerate(pick):
    x = np.load(QC[k]["path"])["x"]
    axs[0, j].imshow(rgb(x)); axs[0, j].set_title(f"{k[0]} {k[1]}  RGB", fontsize=9)
    axs[1, j].imshow(strict_water(x), cmap="gray"); axs[1, j].set_title(f"weak label  {QC[k]['label_ha']:.1f} ha", fontsize=9)
    axs[2, j].imshow(x[..., C["VV"]], cmap="gray", vmin=-25, vmax=0); axs[2, j].set_title("Sentinel-1 VV (dark=water)", fontsize=9)
for a in axs.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(f"{BASE}/review/02_tile_qc.png", dpi=100); plt.show()
print("👀 Row 2 (weak label) should light up the lake only. If it lights up snow/shadows everywhere, tell me before training.")


In [ ]:
#@title 9) Train U-Net and DeepLabv3+ (GPU: Runtime > Change runtime type > T4)
import torch, torch.nn as nn
import segmentation_models_pytorch as smp
dev = "cuda" if torch.cuda.is_available() else "cpu"
assert dev == "cuda", "❌ Turn on a GPU: Runtime > Change runtime type > T4 GPU, then re-run from cell 2"

# Model sees RGB + Sentinel-1 + DEM + slope ONLY. The weak label is built from B8/B11/NDWI, so those bands are
# hidden from the network; otherwise it would just copy the threshold rule. (Say this in your report.)
USE = [C["B2"], C["B3"], C["B4"], C["VV"], C["DEM"], C["SLOPE"]]
EPOCHS = 25

def prep(x):
    z = x[..., USE]; mu = z.mean((0, 1)); sd = z.std((0, 1)) + 1e-6
    return ((z - mu) / sd).astype("float16")

Xtr, Ytr, Xva, Yva, kept = [], [], [], [], 0
for (lid, y), v in QC.items():
    if v["flag"] != "OK" or lid.startswith("BT-"): continue
    x = np.load(v["path"])["x"]
    comp = center_component(clean(strict_water(x), 8), HALF_PX, HALF_PX)   # label = the target lake only
    if comp is None: continue
    X, Y = prep(x), comp.astype("float32")
    if np.isnan(X.astype("float32")).any(): continue
    (Xva if lid in VAL_IDS else Xtr).append(X); (Yva if lid in VAL_IDS else Ytr).append(Y); kept += 1
print(f"train tiles {len(Xtr)} | val tiles {len(Xva)} (held-out lakes {VAL_IDS})")
assert len(Xtr) >= 30 and len(Xva) >= 3, "❌ too few tiles - check cells 7-8"
print(f"water share of pixels: train {np.mean([y.mean() for y in Ytr])*100:.2f}%  val {np.mean([y.mean() for y in Yva])*100:.2f}%")

class TileDS(torch.utils.data.Dataset):
    def __init__(s, X, Y, train, reps=4): s.X, s.Y, s.train, s.reps = X, Y, train, (reps if train else 1)
    def __len__(s): return len(s.X) * s.reps
    def __getitem__(s, i):
        x, y = s.X[i % len(s.X)], s.Y[i % len(s.X)]
        if s.train:
            r, c = np.random.randint(0, 193), np.random.randint(0, 193)     # crop always contains the lake (centre)
            x, y = x[r:r+256, c:c+256], y[r:r+256, c:c+256]
            if np.random.rand() < .5: x, y = x[:, ::-1], y[:, ::-1]
            if np.random.rand() < .5: x, y = x[::-1], y[::-1]
            k = np.random.randint(4); x, y = np.rot90(x, k), np.rot90(y, k)
        return (torch.from_numpy(np.ascontiguousarray(x)).permute(2, 0, 1).float(),
                torch.from_numpy(np.ascontiguousarray(y))[None].float())

dl_tr = torch.utils.data.DataLoader(TileDS(Xtr, Ytr, True), batch_size=8, shuffle=True, num_workers=0)
dl_va = torch.utils.data.DataLoader(TileDS(Xva, Yva, False), batch_size=4, num_workers=0)

def make(arch):
    cls = {"unet": smp.Unet, "deeplab": smp.DeepLabV3Plus}[arch]
    try:    return cls("resnet34", encoder_weights="imagenet", in_channels=len(USE), classes=1)
    except Exception as e:
        print("  (imagenet weights unavailable -> training from scratch)", str(e)[:70])
        return cls("resnet34", encoder_weights=None, in_channels=len(USE), classes=1)

@torch.no_grad()
def evaluate(m, dl):
    m.eval(); tp = fp = fn = 0.0
    for x, y in dl:
        p = (m(x.to(dev)).sigmoid() > .5).float().cpu()
        tp += (p * y).sum().item(); fp += (p * (1 - y)).sum().item(); fn += ((1 - p) * y).sum().item()
    return dict(iou=tp / (tp + fp + fn + 1e-9), dice=2 * tp / (2 * tp + fp + fn + 1e-9),
                precision=tp / (tp + fp + 1e-9), recall=tp / (tp + fn + 1e-9))

dice_l = smp.losses.DiceLoss("binary"); bce_l = nn.BCEWithLogitsLoss()
RESULTS, MODELS = [], {}
for arch in ["unet", "deeplab"]:
    m = make(arch).to(dev)
    xb, yb = next(iter(dl_tr))                      # sanity: one forward pass before the long run
    with torch.no_grad(): o = m(xb.to(dev))
    assert o.shape == yb.shape, f"❌ shape mismatch {o.shape} vs {yb.shape}"
    opt = torch.optim.AdamW(m.parameters(), 5e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
    best, t0 = -1, time.time()
    for ep in range(EPOCHS):
        m.train(); tot = 0
        for x, y in dl_tr:
            x, y = x.to(dev), y.to(dev); opt.zero_grad()
            p = m(x); loss = dice_l(p, y) + bce_l(p, y); loss.backward(); opt.step(); tot += loss.item()
        sched.step(); ev = evaluate(m, dl_va)
        if ev["iou"] > best:
            best = ev["iou"]; torch.save(m.state_dict(), f"{BASE}/models/{arch}.pt"); keep_ev = dict(ev, epoch=ep + 1)
        if ep % 5 == 4 or ep == 0: print(f"{arch:8} ep {ep+1:2d}  loss {tot/len(dl_tr):.3f}  val IoU {ev['iou']:.3f}  Dice {ev['dice']:.3f}")
    RESULTS.append(dict(model={"unet": "U-Net (ResNet34)", "deeplab": "DeepLabv3+ (ResNet34)"}[arch], arch=arch,
                        input_channels="RGB + Sentinel-1 VV + DEM + slope", labels="optical-index weak labels", **keep_ev))
    print(f"  best {arch}: {keep_ev}   ({time.time()-t0:.0f}s)")
    if best < 0.3: print("  ⚠ low IoU - tell me the numbers, labels/threshold may need tuning")

BEST = max(RESULTS, key=lambda r: r["iou"])["arch"]
json.dump(dict(results=RESULTS, best=BEST, val_lakes=VAL_IDS), open(f"{BASE}/out/metrics.json", "w"), indent=1)
print("\n✅ trained. Best model:", BEST)
for r in RESULTS: print(f"{r['model']:24} IoU {r['iou']:.3f}  Dice {r['dice']:.3f}  P {r['precision']:.3f}  R {r['recall']:.3f}")


In [ ]:
#@title 10) Predict every tile -> area time series + polygons + segmentation table
from shapely.geometry import shape, mapping
from shapely.ops import transform as shp_transform
import rasterio.features
from affine import Affine

model = make(BEST).to(dev); model.load_state_dict(torch.load(f"{BASE}/models/{BEST}.pt")); model.eval()
BEST_NAME = [r["model"] for r in RESULTS if r["arch"] == BEST][0]
TS, LATEST, FIRST, SEG = {}, {}, {}, {}
ALLN = {l[0]: l for l in KEEP + BT}

@torch.no_grad()
def predict(x):
    xt = torch.from_numpy(prep(x)).permute(2, 0, 1)[None].float().to(dev)
    return model(xt).sigmoid()[0, 0].cpu().numpy()

for (lid, y), v in sorted(QC.items()):
    if v["flag"] != "OK": continue
    x = np.load(v["path"])["x"]; meta = dict(np.load(v["path"]))
    prob = predict(x); comp = center_component(clean(prob > .5, 8), HALF_PX, HALF_PX)
    if comp is None: continue
    area = float(comp.sum() * 100 / 1e6)                       # 10 m px -> km2
    TS.setdefault(lid, []).append(dict(date=f"{y}-09-30", year=y, area=round(area, 4),
                                       area_weak_label=round(v["label_ha"] / 100, 4)))
    rec = dict(x=x, mask=comp, year=y, meta={k: float(meta[k]) for k in ("x0", "y0", "scale", "epsg")}, area=area,
               conf=float(prob[comp].mean()))
    if lid not in FIRST: FIRST[lid] = rec
    LATEST[lid] = rec
    lab = clean(strict_water(x), 8); lc = center_component(lab, HALF_PX, HALF_PX)
    if lc is not None and y == max(yy for (l2, yy), vv in QC.items() if l2 == lid and vv["flag"] == "OK"):
        inter = (comp & lc).sum(); union = (comp | lc).sum()
        SEG[lid] = dict(lakeId=lid, model=BEST_NAME, iou=round(100 * inter / max(union, 1), 1),
                        diceScore=round(200 * inter / max(comp.sum() + lc.sum(), 1), 1),
                        confidence=round(100 * rec["conf"], 1), timestamp=datetime.datetime.now().isoformat(timespec="seconds"),
                        split="validation" if lid in VAL_IDS else "train")

def polygon_feature(lid, rec, tag):
    m = rec["mask"]; mt = rec["meta"]; tr = Affine(mt["scale"], 0, mt["x0"], 0, -mt["scale"], mt["y0"])
    back = Transformer.from_crs(int(mt["epsg"]), 4326, always_xy=True).transform
    geoms = [shp_transform(back, shape(g)).simplify(2e-6) for g, val in
             rasterio.features.shapes(m.astype("uint8"), mask=m, transform=tr) if val == 1]
    from shapely.ops import unary_union
    g = unary_union(geoms)
    return dict(type="Feature", geometry=mapping(g),
                properties=dict(lakeId=lid, which=tag, year=rec["year"], area_km2=round(rec["area"], 4)))

feats = []
for lid in LATEST:
    feats.append(polygon_feature(lid, FIRST[lid], "first_year")); feats.append(polygon_feature(lid, LATEST[lid], "latest_year"))
json.dump(dict(type="FeatureCollection", features=feats), open(f"{BASE}/out/polygons.geojson", "w"))
json.dump(TS, open(f"{BASE}/out/timeseries.json", "w"), indent=1)
json.dump(list(SEG.values()), open(f"{BASE}/out/segmentation.json", "w"), indent=1)

print(f"✅ {len(TS)} lakes with time series, {len(feats)} polygons, {len(SEG)} segmentation rows")
for lid in list(TS)[:6]:
    print(lid, [(r["year"], r["area"]) for r in TS[lid]])
sv = [s["iou"] for s in SEG.values() if s["split"] == "validation"]
print("IoU vs weak label on held-out lakes (latest year):", sv)


In [ ]:
#@title 11) Risk score (transparent, weighted) + files in the exact shape your frontend expects
W = dict(growth=.30, area=.20, dam_slope=.15, ice_nearby=.15, temp_anom=.10, precip_anom=.10)
DESC = dict(growth="Lake area growth rate (% per year, 2018-2025)", area="Current lake size",
            dam_slope="Steepness of terrain around the lake / moraine rim",
            ice_nearby="Share of snow/ice cover within 1 km (calving / avalanche source)",
            temp_anom="Summer temperature anomaly vs 2018-2024 (ERA5-Land)", precip_anom="Summer precipitation anomaly vs 2018-2024 (ERA5-Land)")

def era5_anom(lat, lng):
    try:
        pt = ee.Geometry.Point([lng, lat]); Y = YEARS[-1]
        col = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR").select(["temperature_2m", "total_precipitation_sum"])
        def jjas(y): return col.filterDate(f"{y}-06-01", f"{y}-10-01")
        cur = ee.Image.cat(jjas(Y).select("temperature_2m").mean(), jjas(Y).select("total_precipitation_sum").sum())
        bt = ee.ImageCollection([jjas(y).select("temperature_2m").mean() for y in range(YEARS[0], Y)]).mean()
        bp = ee.ImageCollection([jjas(y).select("total_precipitation_sum").sum() for y in range(YEARS[0], Y)]).mean()
        v = ee.Image.cat(cur, bt.rename("bt"), bp.rename("bp")).reduceRegion(ee.Reducer.first(), pt, 11132).getInfo()
        return v["temperature_2m"] - v["bt"], (v["total_precipitation_sum"] - v["bp"]) * 1000   # K, mm
    except Exception as e:
        print("  ERA5 failed for", lat, lng, str(e)[:60]); return None, None

raw = {}
for l in KEEP:
    lid = l[0]
    if lid not in TS or len(TS[lid]) < 3: print("skip (need >=3 yearly areas):", lid); continue
    yrs = np.array([r["year"] for r in TS[lid]]); ar = np.array([r["area"] for r in TS[lid]])
    from scipy.stats import theilslopes                     # robust to one noisy year
    slope_fit = theilslopes(ar, yrs)[0]; growth = float(np.clip(100 * slope_fit / max(ar.mean(), 1e-4), -50, 50))
    rec = LATEST[lid]; x, m = rec["x"], rec["mask"]
    dist = ndi.distance_transform_edt(~m)
    ring = (dist > 2) & (dist <= 25)
    dam = float(x[..., C["SLOPE"]][ring].mean())
    big = (dist > 2) & (dist <= 100)
    ndsi = (x[..., C["B3"]] - x[..., C["B11"]]) / (x[..., C["B3"]] + x[..., C["B11"]] + 1e-6)
    ice = float(((ndsi > .4) & (x[..., C["B8"]] > .25))[big].mean())
    ta, pa = era5_anom(SNAP[lid]["lat"], SNAP[lid]["lng"])
    raw[lid] = dict(growth=growth, growth_ok=bool(ar.mean() >= 0.10), area=float(ar[-1]), dam_slope=dam, ice_nearby=ice, temp_anom=ta, precip_anom=pa,
                    elev=float(np.median(x[..., C["DEM"]][m])), prev=float(ar[0]), cur=float(ar[-1]), last_year=int(yrs[-1]))

def norm(key):
    use = lambda r: r[key] is not None and (key != "growth" or r["growth_ok"])   # unreliable growth -> neutral 0.5
    vals = [r[key] for r in raw.values() if use(r)]
    lo, hi = (min(vals), max(vals)) if vals else (0, 1)
    return {k: ((r[key] - lo) / (hi - lo + 1e-9) if use(r) else 0.5) for k, r in raw.items()}
N = {k: norm(k) for k in W}

lakes_out, risk_out = [], []
for l in KEEP:
    lid, name, lat, lng, dist_, basin, *_ = l
    if lid not in raw: continue
    r = raw[lid]; contrib = {k: 100 * W[k] * N[k][lid] for k in W}
    score = float(sum(contrib.values())); level = "LOW" if score < 25 else "MEDIUM" if score < 50 else "HIGH" if score < 75 else "CRITICAL"
    top = sorted(contrib, key=contrib.get, reverse=True)[:2]
    s = SNAP[lid]; sg = SEG.get(lid, {})
    lakes_out.append(dict(id=lid, name=name, district=dist_, basin=basin, region=f"{dist_}, Himachal Pradesh",
        coordinates=dict(lat=round(s["lat"], 5), lng=round(s["lng"], 5)), elevation=round(r["elev"]),
        currentArea=round(r["cur"], 3), previousArea=round(r["prev"], 3), areaChange=round(r["cur"] - r["prev"], 3),
        areaChangePct=round(100 * (r["cur"] - r["prev"]) / max(r["prev"], 1e-4), 1), growthRate=round(r["growth"], 1), growthReliable=r["growth_ok"],
        riskLevel=level, riskScore=round(score), lastUpdated=f"{r['last_year']}-09-30", satelliteSource="Sentinel-1/2 (DL segmentation)",
        priority=lid in PRIORITY))
    risk_out.append(dict(lakeId=lid, riskScore=round(score), riskLevel=level, confidence=sg.get("confidence", 0),
        factors=[dict(name=k, contribution=round(contrib[k], 1), description=DESC[k]) for k in W],
        explanation=f"Score is driven mainly by {top[0]} and {top[1]}. Relative ranking among the monitored lakes.",
        assessedAt=datetime.datetime.now().isoformat(timespec="seconds"), modelVersion=f"HIMNETRA-v1 / {BEST_NAME}"))
lakes_out.sort(key=lambda d: -d["riskScore"])
json.dump(lakes_out, open(f"{BASE}/out/lakes.json", "w"), indent=1)
json.dump(risk_out, open(f"{BASE}/out/risk.json", "w"), indent=1)
print(f"✅ lakes.json ({len(lakes_out)} lakes) + risk.json written")
print(f"{'id':10} {'name':26} {'area km2':>8} {'growth %/yr':>11} {'score':>6} level")
for d in lakes_out: print(f"{d['id']:10} {d['name'][:26]:26} {d['currentArea']:8.3f} {d['growthRate']:11.1f} {d['riskScore']:6d} {d['riskLevel']}")


In [ ]:
#@title 12) Charts, South Lhonak back-test, and download everything
import shutil
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
top = [d["id"] for d in lakes_out[:6]]
for lid in top:
    ax[0].plot([r["year"] for r in TS[lid]], [r["area"] for r in TS[lid]], marker="o", label=lid)
ax[0].set_title("Lake area (km²) from deep-learning segmentation - top-6 risk lakes"); ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)
if "BT-SK-001" in TS:
    b = TS["BT-SK-001"]; ax[1].plot([r["year"] for r in b], [r["area"] for r in b], "r-o")
    ax[1].axvline(2023.75, ls="--", c="k"); ax[1].text(2023.8, max(r["area"] for r in b) * .9, "3-4 Oct 2023\nGLOF", fontsize=8)
    ax[1].set_title("Back-test: South Lhonak lake area (Sikkim)"); ax[1].grid(alpha=.3)
    json.dump(b, open(f"{BASE}/out/backtest.json", "w"), indent=1)
    print("South Lhonak:", [(r["year"], r["area"]) for r in b])
else:
    ax[1].text(.1, .5, "No South Lhonak data (see qc.csv)"); print("⚠ back-test lake missing")
plt.tight_layout(); plt.savefig(f"{BASE}/review/03_results.png", dpi=110); plt.show()

shutil.make_archive("/content/himnetra_outputs", "zip", f"{BASE}/out")
print("Files in out/:", sorted(os.listdir(f"{BASE}/out")))
try:
    from google.colab import files; files.download("/content/himnetra_outputs.zip")
except Exception as e: print("Download from Drive instead:", f"{BASE}/out", e)
print("✅ DONE. Put lakes.json, risk.json, timeseries.json, polygons.geojson, segmentation.json, metrics.json, backtest.json in backend/data/")


In [ ]:
#@title 13) Diagnose the South Lhonak back-test (one column per year)
print("METRICS:", json.dumps(json.load(open(f"{BASE}/out/metrics.json"))["results"], indent=1)[:1000])
bt = sorted(((y, v) for (l, y), v in QC.items() if l == "BT-SK-001"), key=lambda t: t[0])
print({y: v["flag"] for y, v in bt}, "-> years missing from the series were rejected by QC")
fig, axs = plt.subplots(3, len(bt), figsize=(3.3 * len(bt), 10))
for j, (y, v) in enumerate(bt):
    x = np.load(v["path"])["x"]; prob = predict(x)
    comp = center_component(clean(prob > .5, 8), HALF_PX, HALF_PX)
    axs[0, j].imshow(rgb(x)); axs[0, j].set_title(f"{y}  {v['flag']}")
    if comp is not None: axs[0, j].contour(comp, levels=[.5], colors="red", linewidths=.8)
    axs[1, j].imshow(x[..., C["VV"]], cmap="gray", vmin=-25, vmax=0)
    axs[2, j].imshow(prob, vmin=0, vmax=1)
for a in axs.ravel(): a.axis("off")
axs[0, 0].text(-20, 224, "RGB+mask", rotation=90, va="center"); axs[1, 0].text(-20, 224, "S1 VV", rotation=90, va="center")
plt.tight_layout(); plt.savefig(f"{BASE}/review/04_south_lhonak_years.png", dpi=100); plt.show()


In [ ]:
#@title 14) South Lhonak: short windows right before vs right after the 3-4 Oct 2023 outburst
WIN = None
def build_image(roi, year):                      # identical to cell 4, but honours WIN
    start, end = WIN if WIN else (f"{year}-07-15", f"{year}-09-30")
    s2c = (ee.ImageCollection(S2).filterBounds(roi).filterDate(start, end)
           .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60)).map(_cloud_mask))
    names = ["B2", "B3", "B4", "B8", "B11"]
    s2 = ee.Image(ee.Algorithms.If(s2c.size().gt(0), s2c.median().select(names),
                                   ee.Image.constant([0, 0, 0, 0, 0]).rename(names)))
    refl = s2.unmask(0).divide(10000)
    ndwi = refl.normalizedDifference(["B3", "B8"]).rename("NDWI")
    s1c = (ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(roi).filterDate(start, end)
           .filter(ee.Filter.eq("instrumentMode", "IW")).select("VV"))
    vv = ee.Image(ee.Algorithms.If(s1c.size().gt(0), s1c.median(), ee.Image.constant(0))).rename("VV").unmask(0)
    dem = ee.Image("USGS/SRTMGL1_003").rename("DEM"); slope = ee.Terrain.slope(dem).rename("SLOPE")
    return refl.addBands([ndwi, vv, dem, slope]).select(BANDS).toFloat()

s = SNAP["BT-SK-001"]
WINS = [("before  (20 Aug-2 Oct 2023)", ("2023-08-20", "2023-10-02")),
        ("after   (8 Oct-30 Nov 2023)", ("2023-10-08", "2023-11-30")),
        ("1 yr on (20 Aug-2 Oct 2024)", ("2024-08-20", "2024-10-02"))]
fig, axs = plt.subplots(2, 3, figsize=(15, 9)); BA = {}
for j, (tag, w) in enumerate(WINS):
    WIN = w
    try:
        x, _ = with_retry(lambda: fetch(s["lat"], s["lng"], 2023))
        ok = (x[..., C["B3"]] > 0).mean() > .9
        prob = predict(x); comp = center_component(clean(prob > .5, 8), HALF_PX, HALF_PX)
        area = float(comp.sum() * 1e-4) if comp is not None else 0.0
        BA[tag] = dict(window=w, area_km2=round(area, 3), optical_cover=round(float((x[..., C["B3"]] > 0).mean()), 2))
        axs[0, j].imshow(rgb(x)); axs[1, j].imshow(x[..., C["VV"]], cmap="gray", vmin=-25, vmax=0)
        if comp is not None: axs[0, j].contour(comp, levels=[.5], colors="red", linewidths=.8)
        axs[0, j].set_title(f"{tag}\n{area:.2f} km²" + ("" if ok else "  (little optical data!)"), fontsize=9)
    except Exception as e:
        print("window failed:", tag, str(e)[:100])
    axs[0, j].axis("off"); axs[1, j].axis("off")
WIN = None
plt.tight_layout(); plt.savefig(f"{BASE}/review/05_south_lhonak_before_after.png", dpi=100); plt.show()
json.dump(BA, open(f"{BASE}/out/backtest_before_after.json", "w"), indent=1); print(BA)
